# PPY - Class 04: OOP, Generators & Iterators
**Topics:** classes, inheritance, `yield`, iterator protocol, dunder methods

## 🧠 Theory

### Class basics
```python
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self): return f"{self.name} speaks"
    def __str__(self): return f"Animal({self.name})"

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed
    def speak(self): return "Woof!"   # override
```

### Encapsulation
```python
self._protected   # convention: "internal"
self.__private    # name-mangled -> _ClassName__private
```

### Generator
```python
def count_up(n):
    for i in range(n):
        yield i          # pause & return value

list(count_up(5))        # [0,1,2,3,4]
gen = (x**2 for x in range(10))  # generator expression
```

### Iterator protocol
```python
class MyIter:
    def __iter__(self): return self
    def __next__(self):
        if done: raise StopIteration
        return next_value
```

### Key dunder methods
| Method | Triggered by |
|--------|-------------|
| `__str__` | `print(obj)` |
| `__len__` | `len(obj)` |
| `__contains__` | `x in obj` |
| `__iter__` | `for x in obj` |
| `__lt__` | `a < b` |
| `__add__` | `a + b` |
| `__call__` | `obj()` |

## ✏️ Task 1 - Wizard Fight (OOP + generator + status effects)

In [ ]:
import random

class Attack:
    def __init__(self, name, damage):
        self.name   = name
        self.damage = damage

class Wizard:
    def __init__(self, name, hp, strength=1.0, defense=0, agility=0.0):
        self.name     = name
        self.hp       = hp
        self.attacks  = []
        self.strength = strength
        self.defense  = defense
        self.agility  = agility
        self.stunned  = False
        self.burning  = False

    def add_attack(self, atk): self.attacks.append(atk)
    def is_alive(self):        return self.hp > 0

    def take_damage(self, dmg):
        actual  = max(0, dmg - self.defense)
        self.hp = max(0, self.hp - actual)
        return actual

    def attack(self, opponent):
        if self.stunned:
            self.stunned = False
            return f"{self.name} is STUNNED – skips turn!"
        if random.random() < opponent.agility:
            return f"{opponent.name} DODGED {self.name}'s attack!"
        chosen = random.choice(self.attacks)
        dmg    = int(chosen.damage * self.strength * (0.8 + random.random() * 0.4))
        actual = opponent.take_damage(dmg)
        msg    = f"{self.name} casts {chosen.name} -> {actual} dmg | {opponent.name} HP: {opponent.hp}"
        if random.random() < 0.15: opponent.burning = True;  msg += " 🔥"
        if random.random() < 0.10: opponent.stunned = True;  msg += " ⚡"
        return msg

def fight(w1, w2):
    turn = 0
    while w1.is_alive() and w2.is_alive():
        a, d = (w1, w2) if turn % 2 == 0 else (w2, w1)
        yield a.attack(d)
        if d.burning and d.is_alive():
            d.hp = max(0, d.hp - 3)
            yield f"  🔥 Burn: 3 dmg to {d.name} | HP: {d.hp}"
            d.burning = False
        turn += 1
    winner = w1.name if w1.is_alive() else w2.name
    yield f"\n🏆 Winner: {winner}!"

merlin  = Wizard("Merlin",  hp=60, strength=1.3, defense=2)
gandalf = Wizard("Gandalf", hp=55, strength=1.1, defense=1, agility=0.15)
for w in [merlin, gandalf]:
    w.add_attack(Attack("Fireball",  15))
    w.add_attack(Attack("Lightning", 12))
    w.add_attack(Attack("Ice Shard", 10))

print("=== WIZARD DUEL ===")
for event in fight(merlin, gandalf):
    print(event)

## ✏️ Task 2 - Fibonacci sequence iterator

In [ ]:
class FibIterator:
    def __init__(self, nums): self.nums = nums; self.i = 0
    def __iter__(self): return self
    def __next__(self):
        if self.i >= len(self.nums): raise StopIteration
        v = self.nums[self.i]; self.i += 1; return v

class FibonacciSequence:
    def __init__(self, n):
        nums, a, b = [], 0, 1
        for _ in range(n):
            nums.append(a); a, b = b, a + b
        self.nums = nums
    def __iter__(self):   return FibIterator(self.nums)
    def reversed(self):   return FibIterator(self.nums[::-1])

fib = FibonacciSequence(10)
print("Forward:", list(fib))
print("Reverse:", list(fib.reversed()))

## ✏️ Task 3 - Product & Cart with dunder methods

In [ ]:
class Product:
    def __init__(self, name, price, category):
        self.name = name; self.price = price; self.category = category
    def __str__(self):   return f"{self.name} (${self.price:.2f}, {self.category})"
    def __repr__(self):  return f"Product({self.name!r}, {self.price})"
    def __lt__(self, o): return self.price < o.price
    def __eq__(self, o): return self.name == o.name and self.price == o.price

class Cart:
    def __init__(self): self._items = []
    def add(self, p):   self._items.append(p)
    def clear(self):    self._items.clear()
    def __len__(self):        return len(self._items)
    def __contains__(self, p): return p in self._items
    def __iter__(self):       return iter(self._items)
    def __getitem__(self, i): return self._items[i]

    def total_price(self):
        total = 0
        for p in self._items:
            price = p.price
            if p.category == "food":        price *= 0.90
            if p.category == "electronics": price *= 2.0
            total += price
        return total * 0.85

    def __str__(self):
        lines = "\n".join(f"  • {p}" for p in self._items)
        return f"Cart ({len(self)} items):\n{lines}\nTotal: ${self.total_price():.2f}"

cart = Cart()
cart.add(Product("Apple",  0.99,   "food"))
cart.add(Product("Milk",   1.49,   "food"))
cart.add(Product("Laptop", 999.99, "electronics"))
cart.add(Product("Book",   19.99,  "other"))

print(cart)
print(f"Items:    {len(cart)}")
print(f"Cheapest: {min(cart)}")
print(f"Priciest: {max(cart)}")
print(f"Laptop in cart: {Product('Laptop', 999.99, 'electronics') in cart}")

## ✏️ Task 4 - Sum of squares (generator expression)

In [ ]:
total = sum(x**2 for x in range(1, 1_000_001))
print(f"Sum of squares 1..1,000,000 = {total:,}")

## ✏️ Task 5 - Longest word without list

In [ ]:
text = "The quick brown fox jumps over the lazy dog"
longest = max((w.strip('.,!?') for w in text.split()), key=len)
print("Longest word:", longest)

## ✏️ Task 6-11 - Extra generator exercises

In [1]:
import random
from itertools import islice

# Task 6 – flatten nested list
lst = [[1,2,3],[4,5,6],[7,8,9]]
flat = list(x for sub in lst for x in sub)
print("Flat:", flat)

# Task 7 – infinite counter
def count_infinite():
    i = 0
    while True: yield i; i += 1

print("First 5:", list(islice(count_infinite(), 5)))

# Task 8 – bright RGB generator
def rgb_gen():
    while True:
        r, g, b = random.randint(0,255), random.randint(0,255), random.randint(0,255)
        if r + g + b > 400: yield (r, g, b)

print("Bright colors:", list(islice(rgb_gen(), 5)))

# Task 11 – dice
def dice():
    while True: yield random.randint(1, 6)

print("5 dice rolls:", list(islice(dice(), 5)))

Flat: [1, 2, 3, 4, 5, 6, 7, 8, 9]
First 5: [0, 1, 2, 3, 4]
Bright colors: [(136, 174, 134), (99, 248, 142), (248, 136, 61), (95, 247, 178), (177, 116, 125)]
5 dice rolls: [5, 3, 2, 4, 3]


## ✏️ Task 9 - Song & Playlist

In [ ]:
class Song:
    def __init__(self, name, duration, author):
        self.name = name; self.duration = duration; self.author = author
    def __str__(self): return f"{self.name} by {self.author} ({self.duration}s)"

class Playlist:
    def __init__(self): self._songs = []
    def add_song(self, s):    self._songs.append(s)
    def remove_song(self, n): self._songs = [s for s in self._songs if s.name != n]
    def __iter__(self):       return iter(self._songs)
    def __len__(self):        return len(self._songs)
    def __getitem__(self, i): return self._songs[i]
    def long_songs(self):     return (s for s in self._songs if s.duration > 200)

pl = Playlist()
pl.add_song(Song("Bohemian Rhapsody", 354, "Queen"))
pl.add_song(Song("Blinding Lights",   200, "The Weeknd"))
pl.add_song(Song("Hotel California",  391, "Eagles"))
pl.add_song(Song("Shape of You",      180, "Ed Sheeran"))

print("All songs:")
for s in pl: print(" ", s)
print(f"Total: {len(pl)}")
print("Long songs (>200s):")
for s in pl.long_songs(): print(" ", s)

---
## 📋 Cheatsheet - Class 04

### Class skeleton
```python
class MyClass:
    def __init__(self, x): self.x = x
    def __str__(self): return f"MyClass({self.x})"

class Child(MyClass):
    def __init__(self, x, y):
        super().__init__(x); self.y = y
```

### Generator
```python
def gen(n):
    for i in range(n): yield i     # suspends here

for x in gen(5): ...
gen_expr = (x**2 for x in range(10))  # lazy
```

### Iterator protocol
```python
class MyIter:
    def __iter__(self): return self
    def __next__(self):
        if done: raise StopIteration
        return next_value
```

### Dunder reference
```python
__init__  __str__  __repr__  __len__
__iter__  __next__ __contains__ __getitem__
__lt__ __le__ __gt__ __ge__ __eq__
__add__ __sub__ __mul__ __truediv__
__call__  __enter__ __exit__
```

### Infinite generators
```python
from itertools import islice
def count(start=0):
    i = start
    while True: yield i; i += 1

list(islice(count(), 5))   # [0,1,2,3,4]
```